<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [ ]:
dfF = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions AS gsc_impressions_feb
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
dfM = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions AS gsc_impressions_feb
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df_trend=dfF.merge(dfM, on=['client_hash_id', 'content_hash_id'], how='right')

In [ ]:
df_trend.head()

In [ ]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] > 0]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = ((df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb']) / df_trend['gsc_impressions_feb']) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [ ]:
df_trend.head()

In [ ]:
display(df_trend.groupby('trend_dir').agg(
    {
        'gsc_sum_position_mar': ['mean', 'count'],
        'gsc_avg_position_mar': ['mean', 'count'],
    }
))

gsc_sum_position_mar        gsc_avg_position_mar       
                              mean  count                 mean  count
trend_dir                                                            
Mild decline          16969.439737  45998            11.553507  45998
Mild growth           10285.324916  32633            10.300511  32633
Sharp decline          7768.833224  45630            14.454727  45630
Strong growth          4419.623334   9977            12.132022   9977

In [ ]:
"""
gsc_avg_position averages across a page's queries,
but since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),
a page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.
This likely explains some of the scrambled middle-bucket ordering — sum_position,
while confounded by query volume, doesn't suffer from this specific averaging distortion.
"""

"\ngsc_avg_position averages across a page's queries,\nbut since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),\na page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.\nThis likely explains some of the scrambled middle-bucket ordering — sum_position,\nwhile confounded by query volume, doesn't suffer from this specific averaging distortion.\n"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
position_share = df['gsc_sum_position'] / df['gsc_sum_position'].sum()

cap_value = position_share.quantile(0.99)
position_share_capped = position_share.clip(upper=cap_value)

score = -df['trend_pct'] * position_share_capped * 1000
df['score']=score
df['score'] = df['score'] * 1000

In [ ]:
df = df[['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score']].copy()

In [ ]:
conditions = [
    (df['trend_pct'] < negative_mean) & (position_share > position_share.median()),
    (df['trend_pct'] < negative_mean) & (position_share <= position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0) & (position_share > position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0),
    (df['trend_pct'] >= 0) & (position_share > position_share.median()),
]
codes = [
    'strong declining trend with low page position',
    'strong declining trend',
    'mild declining trend with low page position',
    'mild declining trend',
    'low page position',
]
df['reason_code'] = np.select(conditions, codes, default='STABLE')


In [ ]:
df['action'] = np.select(
    [
        df['reason_code'] == 'strong declining trend with low page position',
        df['reason_code'].isin(['strong declining trend', 'mild declining trend with low page position']),
    ],
    ['REFRESH', 'MONITOR'],
    default='SKIP'
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
import os
os.makedirs('work/outputs', exist_ok=True)

output_cols = ['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score', 'reason_code', 'action']
ranked_queue = df.sort_values(by='score', ascending=False)[output_cols]
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

In [ ]:
ranked_queue.head(20)

,content_hash_id,client_hash_id,trend_pct,gsc_sum_position,score,reason_code,action
43124,content_39e19a3ec2d95f9d,client_1a730cb2640a1abf,-99.988147,379794,5473.217340,strong declining trend with low page position,REFRESH
67349,content_c27cc4cb0d258665,client_23a62021009f63c4,-99.958049,260994,5471.569808,strong declining trend with low page position,REFRESH
68141,content_d0633f4187569021,client_23a62021009f63c4,-99.948875,186808,5471.067633,strong declining trend with low page position,REFRESH
266681,content_f0703fc6ae385591,client_a80fca3f171ed1de,-99.911394,550324,5469.015949,strong declining trend with low page position,REFRESH
320178,content_660fe2b474b35a4c,client_fef1a8f436438636,-99.766043,179542,5461.059628,strong declining trend with low page position,REFRESH
93535,content_b49acf92cc1c8c7e,client_3197e6291363b4db,-99.757869,205726,5460.612220,strong declining trend with low page position,REFRESH
251168,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,-99.669771,699631,5455.789850,strong declining trend with low page position,REFRESH
67577,content_c67c7e38bda3567e,client_23a62021009f63c4,-99.546873,236045,5449.062591,strong declining trend with low page position,REFRESH
62794,content_74de5f247659e956,client_23a62021009f63c4,-99.489151,237466,5445.902931,strong declining trend with low page position,REFRESH
46141,content_3bfb3f753bcda98d,client_20259bd6705d81d4,-99.368550,397424,5439.301389,strong declining trend with low page position,REFRESH


For each of the top 20, rows fall into three categories based on what's actually driving the score:

- **Clip-floor** (trend_pct near -99% to -100%): main risk is an unstable/unrepresentative February baseline making the drop look more extreme than it is
- **Large position** (sum_position notably high, near or contributing heavily to rank): main risk is query-volume inflating the sum rather than genuine ranking severity
- **Mid-range** (both signals moderate and proportionate): the most defensible flags, hardest to argue are false positives

**Summary:** 6 of 20 rows (4, 7, 11, 12, 13, 14, 20) lean on a large `sum_position` more than trend severity alone, with row 12 the most extreme (2.1M — several times larger than its neighbors) and the best candidate for manual spot-checking. The remaining rows split between clip-floor cases (main risk: unstable February baseline) and well-balanced mid-range cases, which are the most defensible flags in the list.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
df[df['content_hash_id'] == 'content_164c1f53f13bcee1']
df_trend[df_trend['content_hash_id'] == 'content_39e19a3ec2d95f9d'][['gsc_impressions_feb', 'gsc_impressions_mar', 'trend_pct']]

,gsc_impressions_feb,gsc_impressions_mar,trend_pct
43124,42185.0,5.0,-99.988147


## 4. Weak picks + leakage check

**Which picks look wrong and why?**

Initial hypothesis: rows sitting at the trend_pct clip floor (~-99% to -100%) might be weak picks, since a small/noisy February baseline can make a percentage drop look more dramatic than it really is.

Checked directly against Row 1 (`content_39e19a3ec2d95f9d`, top-ranked pick): Feb impressions = 42,185, Mar impressions = 5, trend_pct = -99.99%. This is a large, reliable baseline with a genuine, dramatic collapse — not a small-denominator artifact. This row is not a weak pick; it's one of the most defensible flags in the queue. The "unstable baseline" risk should be checked per-row, not assumed for the whole clip-floor category.

The stronger weak-pick candidate remains **Row 12** (`content_164c1f53f13bcee1`), whose `gsc_sum_position` (2,098,750) is 3-6x larger than its neighbors in the top 20. This is more likely a query-volume artifact (a page tracked across an unusually large number of queries) inflating its position score, rather than genuinely worse ranking performance. This row should be manually reviewed before acting on it as top priority.

**Leakage check**

This rule uses only two signals:
- `trend_pct` — computed from February vs. March `gsc_impressions`, both historical relative to the working month (March 2026), guarded to `feb_impressions >= 30`, clipped at the 99th percentile
- `gsc_sum_position` — March monthly total, aggregated from daily rows

Neither depends on the sealed June test window (`fact_content_query_90d`, permanently excluded), the `needs_refresh` label definition, or any post-March snapshot fields. Verified directly that the Week 3-excluded leakage-risk columns (`content_updated_date`, `last_optimized_date`, `optimization_eligible_date`) are not present in the working dataframe used for scoring:

```python
excluded = ['content_updated_date', 'last_optimized_date', 'optimization_eligible_date']
[col for col in excluded if col in df.columns]
# -> []
```

No FlyRank product-defined flags (e.g. the actual refresh/CTR-fix flag outputs) were used as inputs — only the underlying raw signals (impressions, position) that inform them, consistent with building an independent baseline rather than replicating an existing flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.